In [5]:
!pip install seaborn
!pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
"""
Coral Bleaching Detection — XGBoost Branch (LOCAL VERSION)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score, f1_score
)
from sklearn.base import clone
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import os

# ────────────────────────────────────────────────────────────
# PATHS (CHANGE ONLY THIS IF NEEDED)
# ────────────────────────────────────────────────────────────
BASE_DIR = r"D:\CoralReef"
DATA_PATH = os.path.join(BASE_DIR, "data.csv")

OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ────────────────────────────────────────────────────────────
# 1. LOAD & CLEAN
# ────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]} samples")

for col in ["SSTA", "TSA", "SSTA_DHW", "TSA_DHWMean",
            "Windspeed", "Temperature_Mean", "Turbidity", "Cyclone_Frequency"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

# ────────────────────────────────────────────────────────────
# 2. ENCODE CATEGORICALS
# ────────────────────────────────────────────────────────────
exposure_map = {"Sheltered": 0, "Sometimes": 1, "Exposed": 2}
df["Exposure_Ord"] = df["Exposure"].map(exposure_map).fillna(1)

ocean_le = LabelEncoder()
df["Ocean_Enc"] = ocean_le.fit_transform(df["Ocean_Name"])

# ────────────────────────────────────────────────────────────
# 3. TARGET
# ────────────────────────────────────────────────────────────
def encode_severity(s):
    s = str(s).lower()
    if "mild" in s: return 0
    if "moderate" in s: return 1
    return 2

df["severity"] = df["Bleaching_condition"].apply(encode_severity)
df["binary"] = (df["severity"] >= 1).astype(int)

print("\nClass Distribution:")
print(df["severity"].value_counts())

# ────────────────────────────────────────────────────────────
# 4. FEATURE ENGINEERING
# ────────────────────────────────────────────────────────────
EL_NINO_YEARS = {1982, 1983, 1987, 1988, 1991, 1992,
                 1997, 1998, 2002, 2003, 2004, 2005,
                 2009, 2010, 2015, 2016, 2023}

df["Is_ElNino"] = df["Date_Year"].isin(EL_NINO_YEARS).astype(int)
df["DHW_max"] = df[["SSTA_DHW", "TSA_DHWMean"]].max(axis=1)
df["TSA_pos"] = df["TSA"].clip(lower=0)
df["Exposure_x_DHW"] = df["Exposure_Ord"] * df["DHW_max"]

FEATURES = [
    "Temperature_Mean",
    "DHW_max",
    "TSA_pos",
    "Windspeed",
    "Exposure_Ord",
    "Exposure_x_DHW",
    "Cyclone_Frequency",
    "Ocean_Enc",
    "Date_Year",
    "Is_ElNino",
]

X = df[FEATURES].copy()
y = df["severity"].values

# ────────────────────────────────────────────────────────────
# 5. SMOTE
# ────────────────────────────────────────────────────────────
from collections import Counter

counter = Counter(y)

# target size = 60% of majority class
target_size = int(0.6 * max(counter.values()))

sampling_strategy = {
    cls: target_size for cls in counter if counter[cls] < target_size
}

smote = SMOTE(
    sampling_strategy=sampling_strategy,
    random_state=42
)
X_res, y_res = smote.fit_resample(X, y)

# ────────────────────────────────────────────────────────────
# 6. MODEL
# ────────────────────────────────────────────────────────────
model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

# ── Cross Validation ────────────────────────────────────────
print("\nCross Validation:")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs = []
for tr, val in cv.split(X_res, y_res):
    m = clone(model)
    m.fit(X_res.iloc[tr], y_res[tr])
    preds = m.predict(X_res.iloc[val])
    accs.append(accuracy_score(y_res[val], preds))

print(f"CV Accuracy: {np.mean(accs):.4f}")

# ── Final Training ──────────────────────────────────────────
model.fit(X_res, y_res)

# ────────────────────────────────────────────────────────────
# 7. EVALUATION
# ────────────────────────────────────────────────────────────
y_pred = model.predict(X)

print("\nClassification Report:")
print(classification_report(y, y_pred))

# ────────────────────────────────────────────────────────────
# 8. PROBABILITIES (FOR FUSION)
# ────────────────────────────────────────────────────────────
probs = model.predict_proba(X)

p_healthy = probs[:, 0]
p_bleached = probs[:, 1] + probs[:, 2]

fusion_df = pd.DataFrame({
    "xgb_p_healthy": p_healthy,
    "xgb_p_bleached": p_bleached
})

fusion_path = os.path.join(OUTPUT_DIR, "xgb_fusion_probs.csv")
fusion_df.to_csv(fusion_path, index=False)

print(f"Fusion file saved: {fusion_path}")

# ────────────────────────────────────────────────────────────
# 9. SAVE MODEL
# ────────────────────────────────────────────────────────────
import joblib

model_path = os.path.join(BASE_DIR, "models", "xgb_model_v3.pkl")
os.makedirs(os.path.dirname(model_path), exist_ok=True)

joblib.dump(model, model_path)

print(f"Model saved: {model_path}")

# ────────────────────────────────────────────────────────────
# 10. PLOT
# ────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
sns.histplot(p_bleached, bins=30)
plt.title("P(Bleached) Distribution")
plt.savefig(os.path.join(OUTPUT_DIR, "xgb_distribution.png"))
plt.close()

print("\n✅ DONE")

Loaded 1252 samples

Class Distribution:
severity
0    560
2    381
1    311
Name: count, dtype: int64

Cross Validation:
CV Accuracy: 0.6476

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.98      0.94       560
           1       0.95      0.80      0.87       311
           2       0.99      0.99      0.99       381

    accuracy                           0.94      1252
   macro avg       0.95      0.92      0.93      1252
weighted avg       0.94      0.94      0.94      1252

Fusion file saved: D:\CoralReef\outputs\xgb_fusion_probs.csv
Model saved: D:\CoralReef\models\xgb_model_v3.pkl

✅ DONE
